# Import the necessary libraries and modules

In [1]:
import torch  # PyTorch library for deep learning
import torch.nn as nn  # Neural network modules
import numpy as np  # NumPy library for numerical computations
import pennylane as qml  # PennyLane for quantum machine learning
import matplotlib.pyplot as plt  # Matplotlib for plotting
import pandas as pd  # Library for data manipulation and analysis  
dtype = torch.float  # Set default tensor data type to float

In [2]:
def Solution1(r, params):
    M = params[0]
    return -(1 - (2*M)/r)

def Solution2(r, params):
    M = params[0]
    return (1 - (2*M)/r)**-1

def test_PINN(NN, domain, params, num_test):
    r = torch.linspace(domain[0], domain[1], num_test).view(-1, 1)
    with torch.no_grad():
        alpha_pred, beta_pred = NN(r)


    g00_pred, g11_pred = -torch.exp(2*alpha_pred), torch.exp(2*beta_pred)
    g00_exact, g11_exact = Solution1(r, params), Solution2(r, params)
    
    l2_error_g00 = (g00_exact - g00_pred)**2
    l2_metric_g00 = torch.sum(l2_error_g00)

    l2_error_g11 = (g11_exact - g11_pred)**2
    l2_metric_g11 = torch.mean(l2_error_g11)

    return l2_metric_g00, l2_metric_g11

In [3]:
M = 15
params = [M]
num_test = 500
domain = [100, 300]

# Classical Neural Network

In [4]:
def init_weights(m):
    """
    Initializes weights and biases for a linear layer.

    Args:
        m (nn.Linear): The linear layer to initialize.

    Notes:
        - Applies uniform weight initialization within the range [-n0, n0].
        - Initializes biases with a constant value of 0.01.
    """
    if isinstance(m, nn.Linear):
        out_sz = torch.tensor(m.out_features)
        n0 = 1.0/torch.sqrt(out_sz)
        m.weight.data.uniform_(-n0, n0)
        m.bias.data.fill_(0.01)

class CNeuralNet(nn.Module):
    """
    Classical neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        output_size (int): Size of the output.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with six linear layer followed by a LogSigmoid activation.

    Notes:
        - The classical neural network architecture consists of six hidden layers.
        - The input layer has 'input_size' neurons.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, output_size):
        super(CNeuralNet, self).__init__()

        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, output_size),
        )

        # self.fnn.apply(init_weights)

    def forward(self, x):
        out = self.fnn(x)
        a = out[:, 0].view(-1, 1)
        b = out[:, 1].view(-1, 1)
        return a, b

In [5]:
# Seeds and hidden sizes
seeds = [14, 42, 86, 195]
hidden_sizes = [10, 30, 50]

# Store results
results = []

for hs in hidden_sizes:
    mses_g00 = []
    mses_g11 = []

    for seed in seeds:
        # Load model
        input_size, hidden_size, output_size = 1, hs, 2
        model_path = f"..\\trained_models\\classical einstein field equations\\{seed}\\Classical model_{seed}_{hs}.pth"
        model = CNeuralNet(input_size, hidden_size, output_size)
        model.load_state_dict(torch.load(model_path))
        model.eval()

        # Evaluate
        with torch.no_grad():
            mse_g00, mse_g11 = test_PINN(model, domain, params, num_test)
            mses_g00.append(mse_g00.item())
            mses_g11.append(mse_g11.item())

    # Compute stats
    mean_g00 = np.mean(mses_g00)
    std_g00 = np.std(mses_g00)
    mean_g11 = np.mean(mses_g11)
    std_g11 = np.std(mses_g11)

    # Store row
    results.append([hs, mean_g00, std_g00, mean_g11, std_g11])

# Create DataFrame
df = pd.DataFrame(results, columns=[
    "Hidden Size",
    "Mean MSE (g00)", "Std. Dev. (g00)",
    "Mean MSE (g11)", "Std. Dev. (g11)"
])

# Format in scientific notation
for col in df.columns[1:]:
    df[col] = df[col].map(lambda x: f"{x:.5e}")

print(df.to_string(index=False))


 Hidden Size Mean MSE (g00) Std. Dev. (g00) Mean MSE (g11) Std. Dev. (g11)
          10    7.50076e-04     6.59699e-04    8.07449e-07     4.39751e-07
          30    3.18135e-04     1.33430e-04    1.21608e-06     1.04041e-06
          50    6.23560e-04     4.37574e-04    2.70455e-06     3.13314e-06


# Quantum Neural Network

In [6]:
# Set the number of qubits to 3
n_qubits = 2
# Create a quantum device with 3 qubits
dev = qml.device("default.qubit", wires=n_qubits)

def RY_layer(w):
    """
    Apply a layer of single-qubit rotations around the Y-axis (RY gates).

    Args:
        w (list): List of rotation angles for each qubit.
    """
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

@qml.qnode(dev)
def quantum_net(input_features, n_qubits, ansatz_depth, feature_weights, ansatz_weights, out_weights):
    """
    Quantum circuit

    Args:
        input_features: Input features.
        n_qubits (int): Number of qubits in the circuit.
        ansatz_depth (int): Depth of the ansatz (number of layers).
        feature_weights (torch.Tensor): List of parameters.
        ansatz_weights (torch.Tensor): List of parameters.
        out_weights (torch.Tensor): List of parameters.

    Returns:
        tuple: Tuple of expectation values for Pauli-Z operators on qubits 0 and 1 as g00 and g11.
    """
    # apply feature map layers
    temp1 = feature_weights[0]*(input_features[0]**feature_weights[1])
    temp2 = feature_weights[2]*(input_features[0]**feature_weights[3])
    qml.RY(temp1, wires=0)
    qml.RY(temp2, wires=1)
    qml.CNOT(wires=[0,1])

    # apply ansatz layers
    for k in range(ansatz_depth):
        RY_layer(ansatz_weights[k])
        qml.CNOT(wires=[0,1])

    # Compute expectation values for Pauli-Z operators
    exp_vals = [qml.expval(out_weights[0]*qml.PauliZ(0)), qml.expval(out_weights[1]*qml.PauliZ(1))]
    return tuple(exp_vals)


class QNeuralNet(nn.Module):
    """
    quantum neural network class.

    Args:
        n_qubits (int): Number of qubits in the circuit.
        feature_depth (int): Depth of the feature map (number of layers).
        ansatz_depth (int): Depth of the ansatz (number of layers).

    Notes:
        - The quantum neural network architecture consists of two qubits.
        - The feature map has 'feature_depth' layers.
        - The ansatz has 'ansatz_depgh' layers.
    """
    def __init__(self, n_qubits, feature_depth, ansatz_depth):
        super(QNeuralNet, self).__init__()
        
        # Set the number qubits, number of ansatz layer and feature map layers
        self.n_qubits = n_qubits
        self.ansatz_depth = ansatz_depth
        self.feature_depth = feature_depth
        
        # Initialize parameters of quantum circuit
        self.feature_weights = nn.Parameter(0.1*torch.randn((2*n_qubits*feature_depth)).view(-1, 1))
        self.ansatz_weights = nn.Parameter(torch.randn(ansatz_depth*n_qubits).view(ansatz_depth, n_qubits))
        self.out_weights = nn.Parameter(torch.FloatTensor(2).uniform_(-1, 1))

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            x (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Scale features by dividing by 100
        x = x/100
        # Initialize an empty tensor for quantum output
        q_out = torch.Tensor(0, 2)

        for elem in x:
            # Apply the quantum circuit (quantum_net) to each input element
            q_out_elem = torch.hstack(tuple(quantum_net(elem, self.n_qubits, self.ansatz_depth, self.feature_weights,self.ansatz_weights, self.out_weights))).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem)) # Concatenate quantum outputs

        # Compute the ansatz solution by applying the arcsine function to the neural network outputs
        out1 = torch.arcsin(q_out[:, 0]).view(-1, 1)
        out2 = torch.arcsin(q_out[:, 1]).view(-1, 1)
        
        return out1, out2

In [7]:
# Seeds and ansatz layers
seeds = [14, 42, 86, 195]
ansatz_layers = [1, 2, 3]

feature_depth = 1


# Store results
results = []

for layer in ansatz_layers:
    mses_g00 = []
    mses_g11 = []

    for seed in seeds:
        # Load model
        model_path = f"..\\trained_models\\quantum einstein field equations\\{seed}\\quantum model_{seed}_{layer}.pth"
        model = QNeuralNet(n_qubits, feature_depth, layer)
        model.load_state_dict(torch.load(model_path))
        model.eval()

        # Evaluate
        with torch.no_grad():
            mse_g00, mse_g11 = test_PINN(model, domain, params, num_test)
            mses_g00.append(mse_g00.item())
            mses_g11.append(mse_g11.item())

    # Compute stats
    mean_g00 = np.mean(mses_g00)
    std_g00 = np.std(mses_g00)
    mean_g11 = np.mean(mses_g11)
    std_g11 = np.std(mses_g11)

    # Store result
    results.append([layer, mean_g00, std_g00, mean_g11, std_g11])

# Create DataFrame
df = pd.DataFrame(results, columns=[
    "Ansatz Depth",
    "Mean MSE (g00)", "Std. Dev. (g00)",
    "Mean MSE (g11)", "Std. Dev. (g11)"
])

# Format in scientific notation
for col in df.columns[1:]:
    df[col] = df[col].map(lambda x: f"{x:.5e}")

print(df.to_string(index=False))


 Ansatz Depth Mean MSE (g00) Std. Dev. (g00) Mean MSE (g11) Std. Dev. (g11)
            1    1.39465e-02     1.68148e-02    2.34815e-03     3.47523e-03
            2    1.09139e-02     1.51738e-02    3.14617e-03     3.93718e-03
            3    1.75493e-03     1.27301e-03    4.46080e-06     3.11691e-06


# Hybrid Quantum Neural Network

In [8]:
# Set the number of qubits to 3
n_qubits = 3
# Create a quantum device with 3 qubits
dev = qml.device("default.qubit", wires=n_qubits)

def RY_layer(w):
    """
    Apply a layer of single-qubit rotations around the Y-axis (RY gates).

    Args:
        w (list): List of rotation angles for each qubit.
    """
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

def entangling_layer(nqubits):
    """
    Apply an entangling layer of CNOT gates to adjacent qubits.

    Args:
        nqubits (int): Number of qubits in use.
    """
    for i in range(0, nqubits - 1):
        qml.CNOT(wires=[i, i + 1])

def init_weights(m):
    """
    Initializes weights and biases for a linear layer.

    Args:
        m (nn.Linear): The linear layer to initialize.

    Notes:
        - Applies uniform weight initialization within the range [-n0, n0].
        - Initializes biases with a constant value of 0.01.
    """
    if isinstance(m, nn.Linear):
        out_sz = torch.tensor(m.out_features)
        n0 = 1.0/torch.sqrt(out_sz)
        m.weight.data.uniform_(-n0, n0)
        # m.bias.data.fill_(0.01)

@qml.qnode(dev)
def hquantum_net(input_features, n_qubits, q_depth, q_weights):
    """
    Quantum circuit as a part of quantum hybrid neural network

    Args:
        input_features (torch.Tensor): Input features.
        n_qubits (int): Number of qubits in the circuit.
        q_depth (int): Number of quantum layers.
        q_weights (torch.Tensor): List of parameters.
    Returns:
        tuple: Tuple of expectation values for Pauli-Z operators on qubits 0, 1, and 2.
    """
    RY_layer(input_features[0])
    entangling_layer(n_qubits)
    
    RY_layer(input_features[1])
    RY_layer(q_weights[0])
    entangling_layer(n_qubits)
    
    RY_layer(input_features[2])
    RY_layer(q_weights[1])
    entangling_layer(n_qubits)
    
    RY_layer(input_features[3])
    RY_layer(q_weights[2])
    entangling_layer(n_qubits)

    exp_vals = [qml.expval(qml.PauliZ(position)) for position in range(0, n_qubits)]
    return tuple(exp_vals)

class HQNeuralNet(nn.Module):
    """
    quantum hybrid neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        n_qubits (int): Number of qubits in the circuit.
        q_depth (int): Number of quantum layers.
        output_size (int): Size of the output.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with a linear layer followed by a Tanh activation.

    Notes:
        - The first classical neural network architecture consists of one hidden layer.
        - The input layer has 'input_size' neurons.
        - The hidden layers has 'hidden_size' neurons with a Tanh activation.
        - The quantum neural network architecture consists of 'n_qubits' and 'q_depth ' layers.
        - The second classical neural network architecture consists of one linear layer.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, n_qubits, q_depth, output_size):
        super().__init__()
        
        # Define the feedforward neural network (fnn)
        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.LogSigmoid(),
            nn.Linear(hidden_size, (q_depth+1)*n_qubits),
        )
        # self.fnn.apply(init_weights)

        #Define the feedforward (n_qubits, output_size) layer as a output layer
        self.output_layer = nn.Linear(n_qubits, output_size)
        # self.output_layer.apply(init_weights)

        # Set the number qubits, number of quantum layers
        self.n_qubits = n_qubits
        self.q_depth = q_depth
        
        # Initialize parameters of quantum circuit
        self.q_weights = nn.Parameter(0.2*torch.randn(q_depth * n_qubits).view(q_depth, n_qubits))

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            data (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Pass the data through the feedforward neural network (self.fnn)
        out = self.fnn(x)
        # Initialize an empty tensor for quantum output
        q_out = torch.Tensor(0, self.n_qubits)

        for elem in out:
            # Apply the quantum circuit (hquantum_net) to each output element
            elem = elem.view((self.q_depth+1), self.n_qubits)
            q_out_elem = torch.hstack(tuple(hquantum_net(elem, self.n_qubits, self.q_depth, self.q_weights))).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem))

        # Pass the output of quantum circuit through the feedforward linear layer
        out = self.output_layer(q_out)
        a = out[:, 0].view(-1, 1)
        b = out[:, 1].view(-1, 1)
        return a, b

In [9]:
seeds = [14, 42, 86, 195]
hidden_sizes = [10, 30, 50]

# Fixed model parameters
input_size, output_size = 1, 2
q_depth = 3      # You use fixed 3-layer quantum part


# Store results
results = []

for hs in hidden_sizes:
    mses_g00 = []
    mses_g11 = []

    for seed in seeds:
        # Load model
        model_path = f"..\\trained_models\\hybrid einstein field equations\\{seed}\\hybrid model_{seed}_{hs}.pth"
        model = HQNeuralNet(input_size, hs, n_qubits, q_depth, output_size)
        model.load_state_dict(torch.load(model_path))
        model.eval()

        # Evaluate
        with torch.no_grad():
            mse_g00, mse_g11 = test_PINN(model, domain, params, num_test)
            mses_g00.append(mse_g00.item())
            mses_g11.append(mse_g11.item())

    # Compute statistics
    mean_g00 = np.mean(mses_g00)
    std_g00 = np.std(mses_g00)
    mean_g11 = np.mean(mses_g11)
    std_g11 = np.std(mses_g11)

    # Add to results
    results.append([hs, mean_g00, std_g00, mean_g11, std_g11])

# Create table
df = pd.DataFrame(results, columns=[
    "Hidden Size",
    "Mean MSE (g00)", "Std. Dev. (g00)",
    "Mean MSE (g11)", "Std. Dev. (g11)"
])

# Format in scientific notation
for col in df.columns[1:]:
    df[col] = df[col].map(lambda x: f"{x:.5e}")

print(df.to_string(index=False))


 Hidden Size Mean MSE (g00) Std. Dev. (g00) Mean MSE (g11) Std. Dev. (g11)
          10    3.78347e-03     5.60853e-03    1.74288e-03     2.78781e-03
          30    3.92746e-03     3.69922e-03    3.26563e-03     5.58726e-03
          50    3.20763e-03     3.60046e-03    2.52760e-05     2.90224e-05
